# Introduction to Data Science 2026

# Week 6: Recap

## Exercise 1 | Linear regression with feature selection

Download the [TED Talks](https://www.kaggle.com/rounakbanik/ted-talks) dataset from Kaggle. Your task is to predict both the ratings and the number of views of a given TED talk. You should focus only on the <span style="font-weight: bold">ted_main</span> table.

1. Download the data, extract the following ratings from column <span style="font-weight: bold">ratings</span>: <span style="font-weight: bold">Funny</span>, <span style="font-weight: bold">Confusing</span>, <span style="font-weight: bold">Inspiring</span>. Store these values into respective columns so that they are easier to access. Next, extract the tags from column <span style="font-weight: bold">tags</span>. Count the number of occurrences of each tag and select the top-100 most common tags. Create a binary variable for each of these and include them in your data table, so that you can directly see whether a given tag (among the top-100 tags) is used in a given TED talk or not. The dataset you compose should have dimension (2550, 104), and comprise of the 'views' column, the three columns with counts of "Funny", "Confusing and "Inspiring" ratings, and 100 columns which one-hot encode the top-100 most common tag columns.


In [1]:
import ast
import json
from collections import Counter
import pandas as pd

df = pd.read_csv(r"C:\Users\mkygh_tm2pfj8\Downloads\ted_main.csv")

def parse(x):
    try:
        return json.loads(x)
    except:
        return ast.literal_eval(x)

ratings = df["ratings"].apply(parse)
tags = df["tags"].apply(parse)

rating_df = pd.DataFrame([
    {r["name"]: r["count"] for r in row} for row in ratings
])[["Funny", "Confusing", "Inspiring"]]

top_tags = [tag for tag, _ in Counter(
    tag for row in tags for tag in row
).most_common(100)]

tag_df = pd.DataFrame([
    {tag: int(tag in row) for tag in top_tags} for row in tags
])

result = pd.concat([df[["views"]], rating_df, tag_df], axis=1)
result.to_csv(r"C:\Users\mkygh_tm2pfj8\Downloads\ted_main_processed.csv", index=False)

print(result.shape)

(2550, 104)


2. Construct a linear regression model to predict the number of views based on the data in the <span style="font-weight: bold">ted_main</span> table, including the binary variables for the top-100 tags that you just created.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = pd.read_csv(r"C:\Users\mkygh_tm2pfj8\Downloads\ted_main_processed.csv")

X = df.drop(columns=["views"])
y = df["views"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R²:", r2_score(y_test, y_pred))

R²: 0.6749469552136259


3. Do the same for the <span style="font-weight: bold">Funny</span>, <span style="font-weight: bold">Confusing</span>, and <span style="font-weight: bold">Inspiring</span> ratings.

In [4]:
X = df.drop(columns=["views", "Funny", "Confusing", "Inspiring"])

for target in ["Funny", "Confusing", "Inspiring"]:
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\nResults for {target}:")
    print("R²:", r2_score(y_test, y_pred))


Results for Funny:
R²: 0.2093054993393717

Results for Confusing:
R²: 0.18953655784818535

Results for Inspiring:
R²: 0.07712709702403331


4. You will probably notice that most of the tags are not useful in predicting the views and the ratings. You should use some kind of variable selection to prune the set of tags that are included in the model. You can use for example classical p-values or more modern [LASSO](https://en.wikipedia.org/wiki/Lasso_(statistics)) techniques. Which tags are the best predictors of each of the response variables?

In [5]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

df = pd.read_csv(r"C:\Users\mkygh_tm2pfj8\Downloads\ted_main_processed.csv")

targets = ["views", "Funny", "Confusing", "Inspiring"]
rating_columns = ["Funny", "Confusing", "Inspiring"]

tag_columns = [col for col in df.columns
    if col not in ["views"] + rating_columns]

for target in targets:
    X = df.drop(columns=[target])
    y = df[target]

    model = make_pipeline(
        StandardScaler(),
        LassoCV(cv=5, max_iter=100000))
    model.fit(X, y)

    coefficients = model.named_steps["lassocv"].coef_
    results = pd.Series(coefficients, index=X.columns)

    selected_tags = results[
        results.index.isin(tag_columns) & (results != 0)
    ].sort_values(key=abs, ascending=False)

    print(f"\nBest tags for predicting {target}:")
    if selected_tags.empty:
        print("No tags selected.")
    else:
        print(selected_tags)

    print("Selected tags:", len(selected_tags))


Best tags for predicting views:
demo             78216.591366
psychology       71318.407572
global issues   -63957.074151
brain            62476.678143
design          -27814.040181
philosophy      -27444.952118
poverty         -24295.621778
life            -23017.958035
creativity      -16127.277221
TEDx            -14554.079650
religion        -14030.092058
potential        12942.411890
children        -12495.837668
robots           11871.926612
politics         -9267.672081
photography       7634.600805
storytelling     -4806.224476
universe          4528.733524
art              -3243.644464
community        -3116.548780
nature            2092.725500
green             -127.295897
dtype: float64
Selected tags: 22

Best tags for predicting Funny:
humor            131.679585
entertainment     28.472364
children          10.625540
creativity         7.348362
dtype: float64
Selected tags: 4

Best tags for predicting Confusing:
philosophy        2.882329
science           2.657768
evolut

5. Produce summaries of your results. Could you recommend good tags – or tags to avoid! – for speakers targeting plenty of views and/or certain ratings?

In [6]:
for target in ["views", "Funny", "Confusing", "Inspiring"]:
    X = df.drop(columns=[target])
    model = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=100000))
    model.fit(X, df[target])
    coefs = pd.Series(model[-1].coef_, index=X.columns)
    tags = coefs[coefs.index.isin(tag_columns)]
    print(target, "\nBest:", tags.nlargest(10).index.tolist(),
          "\nAvoid:", tags.nsmallest(10).index.tolist())

views 
Best: ['demo', 'psychology', 'brain', 'potential', 'robots', 'photography', 'universe', 'nature', 'technology', 'science'] 
Avoid: ['global issues', 'design', 'philosophy', 'poverty', 'life', 'creativity', 'TEDx', 'religion', 'children', 'politics']
Funny 
Best: ['humor', 'entertainment', 'children', 'creativity', 'technology', 'science', 'global issues', 'culture', 'TEDx', 'design'] 
Avoid: ['technology', 'science', 'global issues', 'culture', 'TEDx', 'design', 'business', 'health', 'innovation', 'society']
Confusing 
Best: ['philosophy', 'science', 'evolution', 'physics', 'culture', 'religion', 'brain', 'entertainment', 'math', 'design'] 
Avoid: ['innovation', 'animals', 'health', 'nature', 'Africa', 'oceans', 'robots', 'TEDx', 'history', 'music']
Inspiring 
Best: ['life', 'education', 'mental health', 'happiness', 'creativity', 'global issues', 'business', 'TEDx', 'work', 'entrepreneur'] 
Avoid: ['humor', 'neuroscience', 'demo', 'science', 'economics', 'evolution', 'mind', 'e

**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**

## Exercise 2 | Symbol classification (part 2)

Note that it is strongly recommended to use Python in this exercise. However, if you can find a suitable AutoML implementation for your favorite language (e.g [here](http://h2o-release.s3.amazonaws.com/h2o/master/3888/docs-website/h2o-docs/automl.html) seems to be one for R) then you are free to use that language as well.

Use the preprocessed data from week 3 (you can also produce them using the example solutions of week 3).

1. This time train a *random forest classifier* on the data. A random forest is a collection of *decision trees*, which makes it an *ensemble* of classifiers. Each tree uses a random subset of the features to make its prediction. Without tuning any parameters, how is the accuracy?

2. The amount of trees to use as a part of the random forest is an example of a hyperparameter, because it is a parameter that is set prior to the learning process. In contrast, a parameter is a value in the model that is learned from the data. Train 20 classifiers, with varying amounts of decision trees starting from 10 up until 200, and plot the test accuracy as a function of the amount of classifiers. Does the accuracy keep increasing? Is more better?

In [6]:
# Use this cell for your code

3. If we had picked the amount of decision trees by taking the value with the best test accuracy from the last plot, we would have *overfit* our hyperparameters to the test data. Can you see why it is a mistake to tune hyperparameters of your model by using the test data?

*You can use this markdown cell to write down your thoughts.*

4. Reshuffle and resplit the data so that it is divided in 3 parts: training (80%), validation (10%) and test (10%). Repeatedly train a model of your choosing (e.g random forest) on the training data, and evaluate it’s performance on the validation set, while tuning the hyperparameters so that the accuracy on the validation set increases. Then, finally evaluate the performance of your model on the test data. What can you say in terms of the generalization of your model?

In [8]:
# Use this cell for your code

**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**

## Exercise 3 | TPOT

The process of picking a suitable model, evaluating its performance and tuning the hyperparameters is very time consuming. A new idea in machine learning is the concept of automating this by using an optimization algorithm to find the best model in the space of models and their hyperparameters. Have a look at [TPOT](https://github.com/EpistasisLab/tpot), an automated ML solution that finds a good model and a good set of hyperparameters automatically. Try it on this data, it should outperform simple models like the ones we tried easily. Note that running the algorithm might take a while, depending on the strength of your computer. 

*Note*: In case it is running for too long, try checking if the parameters you are using when calling TPOT are reasonable, i.e. try reducing number of ‘generations’ or ‘population_size’. TPOT uses cross-validation internally, so we don’t need our own validation set.

In [19]:
from tpot import TPOTRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tpot = TPOTRegressor(
    search_space="linear",
    scorers=["r2"],
    population_size=20,
    max_time_mins=10,
    cv=5,
    random_state=42,
    verbose=2
)

tpot.fit(X_train, y_train)

y_pred = tpot.predict(X_test)
print("R²:", r2_score(y_test, y_pred))

Generation: : 1it [11:08, 668.93s/it]
c:\Users\mkygh_tm2pfj8\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\ensemble\_forest.py:2694: FutureWarning: Value `"friedman_mse"` for `criterion` is deprecated and will be removed in 1.11. It maps to `"squared_error"` as both were always equivalent. Use `criterion="squared_error"` to remove this warning.
  warn(
c:\Users\mkygh_tm2pfj8\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\ensemble\_forest.py:1942: FutureWarning: Value `"friedman_mse"` for `criterion` is deprecated and will be removed in 1.11. It maps to `"squared_error"` as both were always equivalent. Use `criterion="squared_error"` to remove this warning.
  warn(
c:\Users\mkygh_tm2pfj8\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\tree\_classes.py:1351: FutureWarning: Value `"friedman_mse"` for `criterion` is deprecated and will be removed in 1.11. It maps to `"squared_error"` as both were always equivalent. Use `criterion="sq

R²: 0.5582853756184813


**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**